# Module 13 Lab — Observability as Governance Evidence

Build and evaluate a synthetic procurement evidence pipeline. The canonical path is deterministic and credential-free: it makes no model, network, collector, shell, cloud, or business-system call.

In [ ]:
from pathlib import Path
import sys
MODULE = Path('curriculum/advanced/13-observability-as-governance-evidence').resolve()
if str(MODULE) not in sys.path:
    sys.path.insert(0, str(MODULE))
from lab import (
    Disposition, EvidenceStore, RetentionClass, TraceEnvelope,
    build_collector_config, build_denied_near_miss_trace,
    build_high_risk_purchase_trace, build_openai_trace_artifacts,
    build_otel_artifacts, build_routine_read_trace, build_tool_manifests,
    choose_retention, compute_governance_metrics, sample_actor, sample_policy,
    verify_evidence_package, verify_trace,
)
print('Course 13 evidence lab loaded; no credentials required.')

## 1. Compare three realistic evidence trajectories

The corpus includes a high-risk approved purchase with a verified effect, a denied external-egress near miss, and a routine read. Every trace uses W3C-sized identifiers, causal parent links, a versioned schema/policy/target binding, and a terminal outcome.

In [ ]:
purchase = build_high_risk_purchase_trace()
near_miss = build_denied_near_miss_trace()
routine = build_routine_read_trace()
traces = (purchase, near_miss, routine)
trajectory = [(e.sequence, e.kind.value, e.parent_span_id, e.decision.value if e.decision else None) for e in purchase.events]
print(trajectory)
assert [row[0] for row in trajectory] == list(range(1, 9))

## 2. Minimize content and validate integrity/completeness

The policy rejects prompt/content/credential fields, allowlists structured attributes, redacts residual secrets and records references/digests rather than raw documents. Each event is HMAC-authenticated, binds the prior event digest, and the envelope binds the target, schema, policy and event list. Integrity does not prove completeness or truth.

In [ ]:
serialized = ''.join(trace.model_dump_json() for trace in traces)
checks = {trace.trace_id: verify_trace(trace) for trace in traces}
print({key: value.model_dump() for key, value in checks.items()})
assert all(result.reconstructable for result in checks.values())
assert 'approved laptop vendors' not in serialized
assert '@example.com' not in serialized

## 3. Fail closed on tampered or partial evidence

The builder will not seal without terminal evidence and enforces exact proposal, approval, authorization, effect and outcome binding. The verifier revalidates an envelope rather than trusting an object that merely has the right fields.

In [ ]:
values = {name: getattr(routine, name) for name in TraceEnvelope.model_fields}
values['chain_head'] = 'f' * 64
forged = TraceEnvelope.model_construct(**values)
failed = verify_trace(forged)
print(failed.model_dump())
assert failed.valid is False
assert failed.reason_codes == ('TRACE_INTEGRITY_INVALID',)

## 4. Govern ingestion, reading, transfer, retention and legal hold

A tenant store accepts only a current writer whose workload/service matches the producer binding, treats exact retry as idempotent, rejects changed content under the same trace ID, and exposes no delete method. Reads require role and purpose and create chained access receipts. Package transfer creates a separate custody receipt; legal hold overrides disposition.

In [ ]:
policy = sample_policy()
store = EvidenceStore('tenant-acme', policy)
writer = sample_actor()
for trace in traces:
    store.ingest(writer, trace)
auditor = sample_actor(roles=frozenset({'auditor'}), purposes=frozenset({'annual_control_audit'}))
package, custody = store.export_package(auditor, purchase.trace_id, purpose_code='annual_control_audit')
custodian = sample_actor(roles=frozenset({'evidence_custodian'}))
store.place_legal_hold(custodian, purchase.trace_id)
plan = store.plan_disposition(custodian, purchase.trace_id, now=purchase.sealed_at)
print({'package': package.package_id, 'package_valid': verify_evidence_package(package), 'access_receipts': len(store.access_receipts), 'custody': custody.receipt_digest[:16], 'disposition': plan.model_dump()})
assert verify_evidence_package(package)
assert plan.disposition == Disposition.RETAIN

## 5. Use tail retention and exact metric populations

High-risk, DENY/ESCALATE, anomaly and error traces are retained independently of the routine sample. Routine selection is deterministic. This is an observability policy—not a substitute for legal/business record retention. Metrics expose exact trace, risk, decision and near-miss denominators.

In [ ]:
retention = {'purchase': choose_retention(purchase, routine_sample_rate=0), 'near_miss': choose_retention(near_miss, routine_sample_rate=0), 'routine': choose_retention(routine, routine_sample_rate=0)}
metrics = compute_governance_metrics(traces)
print({'retention': {k: v.model_dump() for k, v in retention.items()}, 'metrics': metrics.model_dump()})
assert retention['purchase'].retention_class == RetentionClass.GOVERNANCE
assert retention['near_miss'].retention_class == RetentionClass.SECURITY
assert retention['routine'].retain is False

## 6. Review portable and managed tooling boundaries

OpenTelemetry supplies context, signals, OTLP and Collector components; OpenInference/Phoenix, LangSmith and Langfuse add AI analysis; OpenAI provides native agent spans. None independently proves authorization, completeness, integrity or real-world effect. The Collector artifact uses local ingress, removes verbose GenAI messages, applies memory/batch controls and requires TLS export.

In [ ]:
collector = build_collector_config()
manifests = build_tool_manifests()
pipeline = collector['service']['pipelines']['traces/governance']
print({'pipeline': pipeline, 'tools': {k: v.current_status for k, v in manifests.items()}})
assert pipeline['processors'] == ['memory_limiter', 'attributes/governance', 'tail_sampling/governance', 'batch']

## 7. Construct real SDK artifacts without exporting data

The OpenTelemetry path uses a private provider and in-memory exporter. The Agents SDK configuration explicitly excludes sensitive trace data; its trace and custom span are constructed but never started or exported.

In [ ]:
otel = build_otel_artifacts(purchase)
openai = build_openai_trace_artifacts(purchase)
print({'otel_span_count': len(otel['spans']), 'openai_sensitive_data': openai['run_config'].trace_include_sensitive_data, 'openai_trace': openai['trace'].name})
assert len(otel['spans']) == len(purchase.events)
assert openai['run_config'].trace_include_sensitive_data is False

## Exercises and decision review

1. Add a delegated sub-agent event and test authority-chain reconstruction.
2. Model a collector outage where the business effect succeeds but terminal evidence is missing.
3. Alert on unusual bulk reads and test break-glass review.
4. Replace the synthetic shared key with managed asymmetric signatures and protected checkpoints.
5. Map retention classes to your evidence catalog and jurisdiction-specific schedule.

The notebook proves only its named synthetic invariants. It does not prove collector/backend hardening, production completeness, legal sufficiency, or that an external effect occurred.